# 02 — Silver Cleaning & Data Quality

Bronze JSONL → **Spark** Silver Parquet + DQ reports + **DuckDB** validation. **No Gold. No modeling.**

- Primary engine: **PySpark** (`SILVER_ENGINE = "spark"`). Pandas fallback optional.
- Event absence ≠ missing data (e.g. no pit rows → handled in Gold).
- Outliers flagged; only domain-invalid values removed.
- Run after Bronze with the **same** `USE_GOOGLE_DRIVE` setting as notebook 01.


## Colab setup (required every session)

Identical to `00` and `01`: clone, `pip install -e .`, Drive mount, then import `openf1_pipeline`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



## Start Spark


In [ ]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


## Bronze prerequisites


In [ ]:
from pathlib import Path

from openf1_pipeline.config import (
    get_bronze_dir,
    get_data_quality_reports_dir,
    get_manifests_dir,
    get_output_root,
    get_silver_dir,
)
from openf1_pipeline.silver.build_silver import run_silver_cleaning

SILVER_ENGINE = "spark"  # fallback: "pandas"

BRONZE_DIR = get_bronze_dir()
SILVER_DIR = get_silver_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()
MANIFESTS_DIR = get_manifests_dir()

print("OUTPUT_ROOT:", get_output_root())
print("BRONZE_DIR:", BRONZE_DIR)
print("SILVER_DIR:", SILVER_DIR)
print("DATA_QUALITY_REPORTS_DIR:", DATA_QUALITY_REPORTS_DIR)

jsonl_files = list(BRONZE_DIR.rglob("*.jsonl")) if BRONZE_DIR.is_dir() else []
manifest_path = MANIFESTS_DIR / "ingestion_manifest.csv"
row_counts_path = DATA_QUALITY_REPORTS_DIR / "bronze_row_counts.csv"

print(f"Bronze JSONL files found: {len(jsonl_files)}")
print("ingestion_manifest.csv:", manifest_path.exists(), manifest_path)
print("bronze_row_counts.csv:", row_counts_path.exists(), row_counts_path)

if not jsonl_files:
    raise FileNotFoundError(
        f"Bronze data not found at {BRONZE_DIR}. "
        "Run 01_ingestion_bronze.ipynb first with the same USE_GOOGLE_DRIVE setting."
    )


## Run Silver cleaning (Spark-first)


In [ ]:
outputs = run_silver_cleaning(
    bronze_dir=BRONZE_DIR,
    silver_dir=SILVER_DIR,
    data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
    engine=SILVER_ENGINE,
    spark=spark,
)
outputs["summary"]


## DuckDB validation (Silver Parquet)


In [ ]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_silver_with_duckdb,
)

silver_duckdb = validate_silver_with_duckdb(SILVER_DIR)
duckdb_silver_paths = save_duckdb_validation_reports(
    silver_duckdb, DATA_QUALITY_REPORTS_DIR, prefix="silver"
)
duckdb_silver_paths


## Inspect cleaning impact


In [ ]:
import pandas as pd

impact = pd.read_csv(outputs["paths"]["silver_cleaning_impact_summary"])
inventory = pd.read_csv(outputs["paths"]["silver_table_inventory"])
rules = pd.read_csv(outputs["paths"]["silver_cleaning_rules"])

display(inventory)
display(impact[["table_name", "rows_before", "rows_after", "rows_removed", "row_removal_pct"]])
display(rules.head(20))


## Missingness before vs after


In [ ]:
miss_before = pd.read_csv(outputs["paths"]["silver_missingness_before"])
miss_after = pd.read_csv(outputs["paths"]["silver_missingness_after"])

display(miss_before.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))
display(miss_after.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))


## Duplicates & referential integrity


In [ ]:
dups = pd.read_csv(outputs["paths"]["silver_duplicate_report"])
ref = pd.read_csv(outputs["paths"]["silver_referential_integrity_report"])
rejected = pd.read_csv(outputs["paths"]["silver_rejected_records_summary"])

display(dups)
display(ref)
display(rejected)
